# Title + Objective (Markdown)

Deep analysis of top 2–3 models

# Model Comparison — EcoType Forest Cover Classification

## Objective
This notebook compares the previously computed baseline model results and selects the final tuning candidates.

### What this notebook does
- Loads saved baseline comparison scores
- Compares models using macro F1, accuracy, and score stability
- Interprets the top-performing models
- Selects final tuning candidates
- Saves a written tuning-candidate rationale

### Why this notebook exists
Baseline benchmarking has already been completed.
This notebook is used to make the tuning decision explicit, evidence-based, and reproducible.

In [1]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import yaml

In [5]:
project_root = Path.cwd().resolve().parents[0]

config_dir = project_root / "config"
paths_file = config_dir / "paths.yaml"
train_file = config_dir / "train.yaml"

with open(paths_file, "r", encoding="utf-8") as f:
    paths_cfg = yaml.safe_load(f)

with open(train_file, "r", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)

reports_dir = project_root / paths_cfg["artifacts"]["reports_dir"]
model_results_dir = reports_dir

baseline_scores_file = model_results_dir / "baseline_scores.csv"
comparison_summary_file = model_results_dir / "model_comparison_summary.csv"
selection_rationale_file = model_results_dir / "tuning_candidate_rationale.md"

print("baseline_scores_file:", baseline_scores_file)
print("selection_rationale_file:", selection_rationale_file)

baseline_scores_file: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\reports\baseline_scores.csv
selection_rationale_file: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\reports\tuning_candidate_rationale.md


# Load baseline scores

Select best baselines from previous results

In [6]:
baseline_scores_df = pd.read_csv(baseline_scores_file)
baseline_scores_df

,model_name,cv_f1_macro_mean,cv_f1_macro_std,cv_accuracy_mean,cv_accuracy_std,n_features,n_rows,n_folds
0,random_forest,0.911450,0.004864,0.959613,0.000949,20,145890,5
1,extra_trees,0.911337,0.004756,0.954404,0.000572,20,145890,5
2,decision_tree,0.855887,0.005020,0.935774,0.001601,20,145890,5
3,gradient_boosting,0.795543,0.005720,0.869045,0.001017,20,145890,5
4,knn,0.603832,0.007471,0.866468,0.001264,20,145890,5
5,logistic_regression,0.119753,0.000888,0.704353,0.003143,20,145890,5
6,svc_rbf,0.118470,0.000064,0.706621,0.000045,20,145890,5
7,gaussian_nb,0.005150,0.000292,0.015251,0.000078,20,145890,5


In [7]:
comparison_df = baseline_scores_df.sort_values(
    by=["cv_f1_macro_mean", "cv_accuracy_mean"],
    ascending=False,
    ignore_index=True,
).copy()

comparison_df

,model_name,cv_f1_macro_mean,cv_f1_macro_std,cv_accuracy_mean,cv_accuracy_std,n_features,n_rows,n_folds
0,random_forest,0.911450,0.004864,0.959613,0.000949,20,145890,5
1,extra_trees,0.911337,0.004756,0.954404,0.000572,20,145890,5
2,decision_tree,0.855887,0.005020,0.935774,0.001601,20,145890,5
3,gradient_boosting,0.795543,0.005720,0.869045,0.001017,20,145890,5
4,knn,0.603832,0.007471,0.866468,0.001264,20,145890,5
5,logistic_regression,0.119753,0.000888,0.704353,0.003143,20,145890,5
6,svc_rbf,0.118470,0.000064,0.706621,0.000045,20,145890,5
7,gaussian_nb,0.005150,0.000292,0.015251,0.000078,20,145890,5


## Comparison logic

Models are compared using:

- **Primary metric:** cross-validated macro F1
- **Secondary metric:** cross-validated accuracy
- **Stability check:** standard deviation across CV folds

The goal is to identify the strongest and most stable candidates for hyperparameter tuning.

In [8]:
comparison_df["rank_macro_f1"] = comparison_df["cv_f1_macro_mean"].rank(
    ascending=False,
    method="dense"
).astype(int)

comparison_df["rank_accuracy"] = comparison_df["cv_accuracy_mean"].rank(
    ascending=False,
    method="dense"
).astype(int)

comparison_df = comparison_df[
    [
        "rank_macro_f1",
        "rank_accuracy",
        "model_name",
        "cv_f1_macro_mean",
        "cv_f1_macro_std",
        "cv_accuracy_mean",
        "cv_accuracy_std",
        "n_features",
        "n_rows",
        "n_folds",
    ]
]

comparison_df

,rank_macro_f1,rank_accuracy,model_name,cv_f1_macro_mean,cv_f1_macro_std,cv_accuracy_mean,cv_accuracy_std,n_features,n_rows,n_folds
0,1,1,random_forest,0.911450,0.004864,0.959613,0.000949,20,145890,5
1,2,2,extra_trees,0.911337,0.004756,0.954404,0.000572,20,145890,5
2,3,3,decision_tree,0.855887,0.005020,0.935774,0.001601,20,145890,5
3,4,4,gradient_boosting,0.795543,0.005720,0.869045,0.001017,20,145890,5
4,5,5,knn,0.603832,0.007471,0.866468,0.001264,20,145890,5
5,6,7,logistic_regression,0.119753,0.000888,0.704353,0.003143,20,145890,5
6,7,6,svc_rbf,0.118470,0.000064,0.706621,0.000045,20,145890,5
7,8,8,gaussian_nb,0.005150,0.000292,0.015251,0.000078,20,145890,5


In [9]:
comparison_df.to_csv(comparison_summary_file, index=False)
print("Comparison summary saved to:", comparison_summary_file)

Comparison summary saved to: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\reports\model_comparison_summary.csv


## Focused interpretation

The strongest models should combine:
- high macro F1
- high accuracy
- low standard deviation across folds

Models with very low macro F1 despite moderate accuracy are likely over-favoring majority classes and are weaker tuning candidates for this multiclass task.

# Select top tuning candidates

In [10]:
tuning_candidates_df = comparison_df.head(2).copy()
tuning_candidates_df

,rank_macro_f1,rank_accuracy,model_name,cv_f1_macro_mean,cv_f1_macro_std,cv_accuracy_mean,cv_accuracy_std,n_features,n_rows,n_folds
0,1,1,random_forest,0.911450,0.004864,0.959613,0.000949,20,145890,5
1,2,2,extra_trees,0.911337,0.004756,0.954404,0.000572,20,145890,5


# Extract candidate names

In [11]:
tuning_candidate_names = tuning_candidates_df["model_name"].tolist()
tuning_candidate_names

['random_forest', 'extra_trees']

## Holdout-style observations from baseline runs

### Random Forest
- Strongest overall performance
- Best macro F1 among all tested models
- Strong minority-class handling compared with weaker baselines
- High and balanced per-class performance

### Extra Trees
- Very close to Random Forest in macro F1
- Slightly lower accuracy than Random Forest
- Still highly competitive and stable
- Worth tuning as a second high-potential ensemble candidate

### Logistic Regression / SVC
- Collapsed toward dominant classes
- Very poor minority-class recall
- Weak macro F1 despite moderate overall accuracy
- Not strong tuning priorities in the current setup

### GaussianNB
- Extremely poor performance
- Not suitable for this problem in current feature space

# Build selection rationale text

In [12]:
best_row = tuning_candidates_df.iloc[0]
second_row = tuning_candidates_df.iloc[1]

rationale_lines = [
    "# Tuning Candidate Selection Rationale",
    "",
    "## Selected Models",
    "",
    f"- {best_row['model_name']}",
    f"- {second_row['model_name']}",
    "",
    "## Selection Basis",
    "",
    "- Primary metric: cross-validated macro F1",
    "- Secondary metric: cross-validated accuracy",
    "- Stability considered through standard deviation across CV folds",
    "",
    "## Model Comparison Summary",
    "",
    (
        f"- **{best_row['model_name']}** ranked first with "
        f"macro F1 = {best_row['cv_f1_macro_mean']:.6f} "
        f"(std = {best_row['cv_f1_macro_std']:.6f}) and "
        f"accuracy = {best_row['cv_accuracy_mean']:.6f}."
    ),
    (
        f"- **{second_row['model_name']}** ranked second with "
        f"macro F1 = {second_row['cv_f1_macro_mean']:.6f} "
        f"(std = {second_row['cv_f1_macro_std']:.6f}) and "
        f"accuracy = {second_row['cv_accuracy_mean']:.6f}."
    ),
    "",
    "## Why these models move to tuning",
    "",
    "- Both models clearly outperform the remaining baselines on macro F1.",
    "- Both are stable across CV folds, with low standard deviation.",
    "- Both are strong ensemble methods for structured tabular multiclass classification.",
    "- Their performance gap is small enough that tuning could change final ranking.",
    "",
    "## Why other models were not selected",
    "",
    "- Decision tree was materially below the top two and less competitive overall.",
    "- Gradient boosting underperformed the top ensemble models by a clear margin.",
    "- Logistic regression and SVC showed poor macro F1 and minority-class handling.",
    "- GaussianNB was not competitive for this feature space.",
    "",
    "## Recommendation",
    "",
    "- Move **random_forest** and **extra_trees** to the tuning phase.",
    "- Use macro F1 as the primary tuning metric.",
]

In [13]:
rationale_text = "\n".join(rationale_lines)
selection_rationale_file.write_text(rationale_text, encoding="utf-8")

print(rationale_text)
print("\nSaved to:", selection_rationale_file)

# Tuning Candidate Selection Rationale

## Selected Models

- random_forest
- extra_trees

## Selection Basis

- Primary metric: cross-validated macro F1
- Secondary metric: cross-validated accuracy
- Stability considered through standard deviation across CV folds

## Model Comparison Summary

- **random_forest** ranked first with macro F1 = 0.911450 (std = 0.004864) and accuracy = 0.959613.
- **extra_trees** ranked second with macro F1 = 0.911337 (std = 0.004756) and accuracy = 0.954404.

## Why these models move to tuning

- Both models clearly outperform the remaining baselines on macro F1.
- Both are stable across CV folds, with low standard deviation.
- Both are strong ensemble methods for structured tabular multiclass classification.
- Their performance gap is small enough that tuning could change final ranking.

## Why other models were not selected

- Decision tree was materially below the top two and less competitive overall.
- Gradient boosting underperformed the top ensemble

## Final takeaway

The model comparison phase confirms that the strongest tuning candidates are:

- **Random Forest**
- **Extra Trees**

Next phase:
- define parameter grids
- tune both models using macro F1
- compare tuned results against the untuned baselines